In [1]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


In [2]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee # Google Earth Engine API
import pandas as pd # Data manipulation
import geopandas as gpd # Geospatial data manipulation (builds on pandas)
import folium # <--- NEW: For interactive mapping in notebooks
from configs.regions import kenyan_coast_roi # Import your defined region from config

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
All core libraries imported and GEE initialized.


In [ ]:
# Cell 3: Verify Region of Interest (ROI) Import and Display Map
print("Kenyan Coastal ROI imported and ready.")

# Convert the GEE Geometry to a GeoJSON format that Folium can understand
# GEE's .getInfo() method returns a GeoJSON-like dictionary
roi_geojson = kenyan_coast_roi.getInfo()

# Get the centroid (center point) of your ROI for map initialization
# Note: GEE Geometry centroid returns [lon, lat], Folium expects (lat, lon)
centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat = centroid_coords[1]
center_lon = centroid_coords[0]

# Create a Folium map centered on your ROI
# The 'tiles' argument specifies the base map layer
m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

# Add your ROI polygon to the Folium map
folium.GeoJson(
    roi_geojson,
    name='Kenyan Coastal ROI',
    style_function=lambda x: {
        'fillColor': '#f08080', # Light red fill
        'color': 'red',        # Red border
        'weight': 3,
        'fillOpacity': 0.6
    }
).add_to(m)

# Add a layer control to the map (allows toggling layers)
folium.LayerControl().add_to(m)

print("Interactive map of ROI displayed below:")
# Display the map in the notebook
m

Kenyan Coastal ROI imported and ready.
Interactive map of ROI displayed below:


In [4]:
# Cell 4: Access and Filter Global Mangrove Watch (GMW) Data (Using LANDSAT/MANGROVE_FORESTS)
print("--- Accessing LANDSAT/MANGROVE_FORESTS Data ---")

# Access the ImageCollection
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")

# Filter the collection to your region of interest and select a recent image.
# We'll take the latest available image in the collection that intersects our ROI.
# GEE ImageCollections usually have 'system:time_start' property.
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first() # Get the most recent image

if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# Clip the selected image to the ROI
mangroves_in_roi = recent_mangrove_image.clip(kenyan_coast_roi)

# The LANDSAT/MANGROVE_FORESTS collection typically uses values to denote mangrove presence.
mangrove_mask = mangroves_in_roi.gt(0) # Assuming anything > 0 indicates mangrove
mangroves_filtered = mangroves_in_roi.updateMask(mangrove_mask)

print("LANDSAT/MANGROVE_FORESTS data accessed and filtered.")
print(f"Year of selected image: {ee.Date(recent_mangrove_image.get('system:time_start')).format('YYYY').getInfo()}")
# print(f"Filtered mangroves info (snippet): {str(mangroves_filtered.getInfo())[:200]}...") # Optional: for debug

# --- Visualize Filtered Mangroves on Folium Map ---
mangrove_vis_params = {
    'min': 0, 'max': 1,
    'palette': ['white', 'green'] # Non-mangrove: white, Mangrove: green
}

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

m_gmw = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

roi_geojson = kenyan_coast_roi.getInfo()
folium.GeoJson(
    roi_geojson,
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}
).add_to(m_gmw)

map_id_dict = mangroves_filtered.getMapId(mangrove_vis_params)
folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name=f'Mangrove Forests {ee.Date(recent_mangrove_image.get("system:time_start")).format("YYYY").getInfo()}'
).add_to(m_gmw)

folium.LayerControl().add_to(m_gmw)
m_gmw

--- Accessing LANDSAT/MANGROVE_FORESTS Data ---
LANDSAT/MANGROVE_FORESTS data accessed and filtered.
Year of selected image: 2000
